In [1]:

# Imports and Tools

import json
import re

def calculator(expression):
    """
    Safely evaluates a math expression.
    Using eval is normally dangerous but we strip it down here for the assignment.
    """
    try:
        # Just keeping it simple and striping out weird spaces
        clean_expr = expression.strip()
        # Evaluate the math!
        result = eval(clean_expr, {"__builtins__": None}, {})
        return result
    except Exception as e:
        # returning an error string if someone types 'calculate hello'
        return f"Error computing math: {str(e)}"

def extract_keywords(text):
    """
    Pulls out the longest words from a text as pseudo-keywords.
    """
    # split by spaces and strip puntuation
    words = re.findall(r'\b\w+\b', text)
    if not words:
        return []

    # filter words longer than 5 chars, convert to lowercaes to normalize
    keywords = [w.lower() for w in words if len(w) > 5]
    # return unique keywords only by casting to a set then back to a list
    return list(set(keywords))

print("Tools loaded successfully!")



Tools loaded successfully!


In [3]:

#  Main Agent Pipeline (Task-Routing)

def agent(query):
    """
    The main routing logic.
    Takes a query, checks patterns, routes to the right tool, and returns strict JSON.
    """
    # always convert to lowercase first so we dont miss capitalized words!
    q_lower = query.lower()

    response_payload = {
        "type": "error",
        "result": "Something went wrong in the routing logic."
    }

    try:
        # Route 1: Calculator Tool
        if "calculate" in q_lower:
            # Try to grab whatever comes after the word 'calculate'
            # e.g. "calculate 5 + 5" -> " 5 + 5"
            parts = q_lower.split("calculate")
            if len(parts) > 1:
                math_expr = parts[1].strip()
                if math_expr:
                    calc_result = calculator(math_expr)

                    # check if the calculator returned an error string
                    if isinstance(calc_result, str) and calc_result.startswith("Error"):
                        response_payload = {"type": "error", "result": calc_result}
                    else:
                        response_payload = {"type": "calculation", "result": calc_result}
                else:
                    response_payload = {"type": "error", "result": "No math expression provided after 'calculate'."}
            else:
                 response_payload = {"type": "error", "result": "Malformed calculate command."}

        # Route 2: Keyword Extractor Tool
        elif "keywords" in q_lower:
            # pass the whole original query in just in case capitalization matters
            # for the keyword extraction (even tho our tool lowercases it later anyway)
            extracted = extract_keywords(query)
            response_payload = {"type": "keywords", "result": extracted}

        # Route 3: Fallback / General Conversation
        else:
            response_payload = {
                "type": "general",
                "result": "I am a simple agent. I can only 'calculate' math or extract 'keywords'. Please try rephrasing!"
            }

    except Exception as e:
        # Catch-all for any weird unexpected pipeline failures
        response_payload = {
            "type": "error",
            "result": f"Agent crashed: {str(e)}"
        }

    # Format the final response package as cleanly structured JSON string
    # Ensuring it matches the schema: {"type": ..., "result": ...}
    return json.dumps(response_payload, indent=2)

print("Agent logic mapped!")

Agent logic mapped!


In [4]:

#  Validation Check (Automated Arrays)

print("--- Running Automated Array Check ---")

# Automated test strings to validate the routing logic
test_queries = [
    "Can you calculate 25 * 4 + 10?",
    "Please extract keywords from this incredibly fascinating and wonderful sentence.",
    "Hello there, how are you doing today?",
    "calculate 10 / 0", # testing math error handling
    "extract keywords from dog cat bird", # testing no long words
]

for tq in test_queries:
    print(f"\nUser Query: '{tq}'")
    output = agent(tq)
    print(output)



--- Running Automated Array Check ---

User Query: 'Can you calculate 25 * 4 + 10?'
{
  "type": "error",
  "result": "Error computing math: invalid syntax (<string>, line 1)"
}

User Query: 'Please extract keywords from this incredibly fascinating and wonderful sentence.'
{
  "type": "keywords",
  "result": [
    "please",
    "keywords",
    "extract",
    "wonderful",
    "sentence",
    "fascinating",
    "incredibly"
  ]
}

User Query: 'Hello there, how are you doing today?'
{
  "type": "general",
  "result": "I am a simple agent. I can only 'calculate' math or extract 'keywords'. Please try rephrasing!"
}

User Query: 'calculate 10 / 0'
{
  "type": "error",
  "result": "Error computing math: division by zero"
}

User Query: 'extract keywords from dog cat bird'
{
  "type": "keywords",
  "result": [
    "keywords",
    "extract"
  ]
}


In [5]:

# CELL 4: Interactive Mode

# Interactive while loop as requested in instruction 8
print("\n" + "="*40)
print("--- Starting Interactive Mode ---")
print("Type 'exit' or 'quit' to stop.")

while True:
    user_input = input("\nEnter query: ")
    if user_input.lower() in ['exit', 'quit']:
        print("Exiting agent. Bye!")
        break

    if not user_input.strip():
        continue

    result_json = agent(user_input)
    print(result_json)


--- Starting Interactive Mode ---
Type 'exit' or 'quit' to stop.

Enter query: exit
Exiting agent. Bye!
